In [ ]:
import pickle, matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scipy.stats
from mpl_toolkits.axes_grid1 import make_axes_locatable

import mne
from mne.channels import find_ch_adjacency
from mne.datasets import sample
from mne.stats import combine_adjacency, spatio_temporal_cluster_test
from mne.viz import plot_compare_evokeds

import os
from os.path import join
from itertools import combinations
from scipy.stats import ttest_ind
from mne.stats import f_mway_rm, f_threshold_mway_rm


%gui qt
mne.set_log_level(verbose='WARNING')
matplotlib.use('Qt5Agg')

In [ ]:
#========= Parameters =========#
# STC parameters
SNR = 3 # Usually the rule of thumb has been to set to 3 for ANOVAs, 2 for single trial analyses. 
method = "dSPM"
fixed = False # set to True if you want signed data. this will make this command run: mne.convert_forward_solution(fwd, surf_ori=True)
n_jobs = 4
lambda2 = 1.0 / SNR ** 2.0

#===========================#
expt = 'EXPT' # experiment name as written on the -raw.fif
ROOT = f'/path/to/{expt}' # change path to where your data is stored
os.chdir(ROOT)
# recommended folder structure.
epochs_dir = join(ROOT, 'epochs/')
evokeds_dir = join(ROOT, 'evokeds/')
subjects_dir = join(ROOT,'mri/')
raw_dir = join(ROOT, 'raw/')
meg_dir = join(ROOT, 'meg/')
log_dir  = join(ROOT, 'logs')
stc_dir = join(ROOT, 'stc/')

excluded = ['R0000']
## Use this when processing all the subjects
subjects = [i[:5] for i in os.listdir(raw_dir) if i.startswith('R') and i[:5] not in excluded and not i.endswith(".fif")] # List of subjects, can also be added manually
## Use this when processing some of the subjects
print('N =', len(subjects))
print(subjects)

colors = {
    'CONDITION1': 'tab:blue',
    'CONDITION2': 'tab:orange',
    'CONDITION3': 'tab:green',
    'CONDITION4': 'tab:red'
}

conditions = list(colors.keys())

# permutation test vars 
tmin = 0    # length of stcs
tmax = 0.8  # length of stcs
toi_min = 0.1   # length of test
toi_max = 0.5   # length of test
tail = 1
p_thresh = 0.05
n_permutations = 10000
times = np.arange(int(tmin * 1000), int(tmax * 1000 + 1), 1)

n_times = len(times)
n_cond = len(conditions)
n_subj = len(subjects)
n_hemisources = 2562 # 5124 source space with ico-4
hemi = 'lh' # can be 'rh'

In [ ]:
### adjacency matrix
src_fname = os.path.join(subjects_dir, 'fsaverage', 'bem', 'fsaverage-ico-4-src.fif')
src = mne.read_source_spaces(src_fname)

adjacency = None

if hemi == 'lh':
    adjacency = mne.spatial_src_adjacency(src[:1]) # lh: src[:1], rh: src[1:]
elif hemi == 'rh':
    adjacency = mne.spatial_src_adjacency(src[1:]) # lh: src[:1], rh: src[1:]
else:
    adjacency = mne.spatial_src_adjacency(src)

In [ ]:
annot_name = 'PALS_B12_Lobes'
lobes = mne.read_labels_from_annot('fsaverage', annot_name, subjects_dir = subjects_dir, hemi = 'lh') # cannot use 'bh' here
label_name = f'LOBE.TEMPORAL-lh' # for example
temporal = [label for label in lobes if label.name==label_name][0]

# vertex indices in the ROI
hemi_idx    = np.arange(0, n_hemisources, 1)
roi_idx     = temporal.get_vertices_used(vertices=hemi_idx)

#look for vertex indices NOT in the ROI for spatial_exclude
diff_idx    = np.setdiff1d(hemi_idx, roi_idx)
spatial_exclude = diff_idx

In [ ]:
def get_STC(subj, cond):
    stc_fname = os.path.join(stc_dir, '%s', '%s_%s_dSPM') % (cond, subj, cond)
    stc = mne.read_source_estimate(stc_fname, subject='fsaverage')
    return stc

def stat_fun(*args):
    # Return only the F-values
    return f_mway_rm(np.swapaxes(args, 1, 0), factor_levels=factor_levels,
                     effects=effects, return_pvals=True)[0]

In [ ]:
stcs = [] # store source estimate object
data = [] # get data from stc
X = [] # what the test runs on
for c, cond in enumerate(conditions):
    data_tmp = np.empty((n_subj, n_hemisources, n_times))
    print('Reading in STCs for condition %s' % cond)
    for s, subj in enumerate(subjects):
        stc = get_STC(subj, cond)
        stcs.append(stc)
        idx = s+c*n_subj
        subj_stc = stcs[idx]
        # transpose the matrix into: subjects x times x sources
        data_tmp[s,:,:] = subj_stc.data[:n_hemisources]
    data.append(np.transpose(data_tmp, [0, 2, 1]))
    
    toi_min_idx = np.squeeze(np.where(times==toi_min))
    toi_max_idx = np.squeeze(np.where(times==toi_max))
    toi = np.arange(toi_min, toi_max+1, 1)
    toi_idx = np.arange(toi_min_idx, toi_max_idx+1, 1)
    X.append(data[c][:,toi_idx,:n_hemisources])

print("Time window of analysis: ", toi_min, " to ", toi_max)

In [ ]:
# Estimating F-test threshold
factor_levels = [2,2,2] # shape of the condition contrast (3 factors, 2 levels each)
effects = ['A:B:C'] # A - main effect of A, B - main effect of B, A:B - interaction effect
f_thresh = f_threshold_mway_rm(len(subjects), factor_levels, effects, p_thresh)

In [ ]:
print("Shape of data: ", X[0].shape)
print("Conditions: ", conditions)
print("Launching clustering test for effect:", effects)
print("Threshold:", f_thresh)

F_obs, clusters, clusters_pvals, h0 = clu = \
    mne.stats.spatio_temporal_cluster_test(X, 
                                           tail = tail,
                                           threshold = f_thresh,
                                           stat_fun = stat_fun,
                                           n_permutations = n_permutations,
                                           adjacency = adjacency,
                                           spatial_exclude = spatial_exclude,
                                           out_type='indices',
                                           n_jobs = -1)

# Save results
stats_dir = join(ROOT, 'stats')
if not os.path.exists(stats_dir):
    os.mkdir(stats_dir)

pickle_fname = os.path.join(stats_dir, 
                            'stats_%s_%s-%s_%s_%s_%s_%s.pickled' 
                            % (effects, str(toi_min), str(toi_max), label_name, hemi))
with open(pickle_fname, "wb") as open_file:
    pickle.dump(clu, open_file)

# print any cluster below p=0.05
p_thresh = 0.05     # significance threshold
print(clusters_pvals)
print(np.where(clusters_pvals < p_thresh)[0])
good_cluster_inds = np.where(clusters_pvals < p_thresh)[0]